# The Intent Gap — Free Pilot (Stage A regex filter on WildChat)

**What this does:** streams a 10,000-conversation sample from WildChat-1M, runs the Stage A frustration-signal regex filter, prints the funnel, and saves 30 example matches for hand review.

**What this needs:** a Google account (Gmail). Nothing else. No installs, no API keys, no money.

**How to run:** `Runtime` → `Run all` (or hit ▶️ on each cell in order).

**Time:** ~3–5 minutes total.

**Output:** two files saved to your Colab session — `pilot_funnel.json` and `pilot_examples.jsonl`. Download them via the left-side files panel and drop into the GitHub repo's `data/` folder.

## 1. Install dependencies

In [ ]:
!pip install -q datasets

## 2. Define the repair-signal regex

These phrases mark a user repairing a previous assistant response — i.e., the assistant likely missed user intent. This list is the v1 seed; expand later.

In [ ]:
import re

REPAIR_PHRASES = [
    r"no,?\s+i\s+meant",
    r"that(?:'s|\s+is)\s+not\s+what\s+i\s+(?:asked|wanted|meant)",
    r"you\s+misunderstood",
    r"you\s+missed\s+the\s+point",
    r"let\s+me\s+rephrase",
    r"let\s+me\s+clarify",
    r"to\s+be\s+clear,?\s+i\s+(?:want|meant|need)",
    r"actually,?\s+i\s+(?:want|meant|need)",
    r"what\s+i\s+actually\s+(?:need|want|meant)",
    r"the\s+question\s+was",
    r"you\s+didn'?t\s+answer\s+my\s+question",
    r"read\s+the\s+prompt\s+again",
    r"this\s+is\s+(?:wrong|incorrect|not\s+what\s+i'?m\s+looking\s+for)",
    r"you\s+got\s+it\s+wrong",
    r"(?:not\s+useful|useless|unhelpful)",
    r"i\s+think\s+you\s+misunderstood",
    r"that\s+wasn'?t\s+the\s+question",
]

REPAIR_RE = re.compile('|'.join(f'({p})' for p in REPAIR_PHRASES), flags=re.IGNORECASE)

def has_repair_signal(text):
    return bool(REPAIR_RE.search(text or ''))

# Quick sanity check
print(has_repair_signal('No, I meant the second one.'))   # True
print(has_repair_signal('Can you book a flight?'))         # False

## 3. Stream WildChat from Hugging Face

WildChat-1M (CC BY-NC). We use streaming so we don't download all 25GB.

In [ ]:
from datasets import load_dataset

# Streaming mode: pulls rows on demand, no full download.
# Use the non-toxic subset to skip moderation-flagged content.
ds = load_dataset("allenai/WildChat-1M", split='train', streaming=True)

print('Dataset stream opened.')

## 4. Run the Stage A filter on a 10,000-conversation sample

In [ ]:
import json
from collections import Counter

SAMPLE_SIZE = 10_000

n_total = 0
n_english = 0
n_long_enough = 0
n_repair_signal = 0
phrase_counter = Counter()
candidates = []

for row in ds:
    if n_total >= SAMPLE_SIZE:
        break
    n_total += 1

    # Filter to English
    if row.get('language', '').lower() != 'english':
        continue
    n_english += 1

    # Need ≥4 turns (2 user + 2 assistant)
    turns = row.get('conversation', [])
    if len(turns) < 4:
        continue
    n_long_enough += 1

    # Get user turn 2 (third entry, since 0=user, 1=assistant, 2=user)
    try:
        user_t1 = turns[0]['content']
        asst_t1 = turns[1]['content']
        user_t2 = turns[2]['content']
    except (KeyError, IndexError):
        continue

    if not has_repair_signal(user_t2):
        continue
    n_repair_signal += 1

    # Track which phrase matched
    m = REPAIR_RE.search(user_t2)
    if m:
        phrase_counter[m.group(0).lower()] += 1

    candidates.append({
        'conv_id': row.get('conversation_hash', ''),
        'user_t1': user_t1[:1500],     # truncate for display
        'asst_t1': asst_t1[:1500],
        'user_t2': user_t2[:1500],
        'matched_phrase': m.group(0) if m else '',
    })

print(f'Total streamed:       {n_total:>6}')
print(f'English:              {n_english:>6}')
print(f'≥4 turns:             {n_long_enough:>6}')
print(f'Repair signal hit:    {n_repair_signal:>6}')
print(f'Repair signal rate:   {n_repair_signal / max(n_long_enough,1) * 100:.2f}% of ≥4-turn English convs')

## 5. Top matched phrases

In [ ]:
for phrase, count in phrase_counter.most_common(15):
    print(f'{count:>4}   {phrase}')

## 6. Show 5 example matches

In [ ]:
for i, c in enumerate(candidates[:5]):
    print(f'--- Example {i+1} ---')
    print(f'Matched phrase: {c["matched_phrase"]}')
    print(f'\nUSER #1:\n{c["user_t1"][:500]}')
    print(f'\nASSISTANT #1:\n{c["asst_t1"][:500]}')
    print(f'\nUSER #2 (repair):\n{c["user_t2"][:500]}')
    print('\n' + '='*70 + '\n')

## 7. Save outputs to files

Files appear in the left-side **Files** panel of Colab. Right-click → Download to grab them onto your laptop, then upload to the GitHub repo's `data/` folder via your browser.

In [ ]:
# Funnel summary
funnel = {
    'sample_size': n_total,
    'english': n_english,
    'at_least_4_turns': n_long_enough,
    'repair_signal_hit': n_repair_signal,
    'rate_per_4turn_english': round(n_repair_signal / max(n_long_enough, 1) * 100, 3),
    'top_phrases': phrase_counter.most_common(20),
    'source': 'allenai/WildChat-1M (streamed sample)',
    'stage': 'A (regex only) — no LLM judge yet',
}

with open('pilot_funnel.json', 'w') as f:
    json.dump(funnel, f, indent=2)

# Save 30 example matches
with open('pilot_examples.jsonl', 'w') as f:
    for c in candidates[:30]:
        f.write(json.dumps(c) + '\n')

print('Saved: pilot_funnel.json and pilot_examples.jsonl')
print('Look in the Files panel on the left to download.')

## 8. What to do next

1. Download `pilot_funnel.json` and `pilot_examples.jsonl` from the Files panel on the left.
2. Upload both to your GitHub repo, into the `data/` folder.
3. Eyeball the 30 examples. Which ones are real intent-gap failures vs. false positives (e.g., the user's repair was triggered by a refusal, not a real intent miss)?
4. Tell Claude what you saw. We'll use the funnel numbers in the paper's Findings section, refine the phrase list based on false positives, and iterate.

**Stretch:** bump `SAMPLE_SIZE` from 10,000 to 100,000 (cell 4) for a bigger run. Will take ~30 minutes.